# 00 - Setup & smoke test\n\nVerifies GPU, installs deps, and runs a 3-question smoke generation with the primary model. **Expected:** 3 traces with parsed answers.

In [ ]:
# @title Setup (run once per session)
import os, sys
ROOT = "/content/mini_project"
if not os.path.isdir(ROOT):
    raise SystemExit(
        "Upload the mini_project folder to Colab first: Files panel -> Upload -> "
        "mini_project.zip, then unzip: !unzip -q mini_project.zip -d /content/  "
        "(or git clone your repo into /content/mini_project)"
    )
os.chdir(ROOT)
sys.path.insert(0, ROOT)  # project root, so `src` imports as a package (src/*.py use relative imports)
get_ipython().system("pip install -q -r requirements.txt")
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# <-- SET ME: path to your `data` folder in Google Drive, e.g.
#     "/content/drive/MyDrive/mini_project/data". Leave "" to use the local
#     ./data folder inside this repo instead (no Drive mount).
DATA_DIR = ""
if DATA_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["MINI_PROJECT_DATA_DIR"] = DATA_DIR
    print("data dir ->", DATA_DIR)
# HF token: only needed for GATED models (not for the primary Qwen medical model)
import getpass
_hf = getpass.getpass("HF token (press Enter to skip): ")
if _hf:
    os.environ["HF_TOKEN"] = _hf
print("setup OK")


In [ ]:
from src.common import load_config, load_prompt_config, load_model_and_tokenizer, set_seed
from src.inference import build_prompt, generate_and_score, parse_answer_letter, letter_token_ids
cfg = load_config()
pcfg = load_prompt_config()
set_seed(cfg["seed"])
model_id = cfg["model"]["primary"]
print("model:", model_id)
model, tok = load_model_and_tokenizer(model_id, dtype=cfg["model"]["dtype"])
# tiny smoke set: first 3 MedQA questions
from src.data_loading import load_medqa
items = load_medqa(n=3, seed=cfg["seed"])
for it in items:
    rec = generate_and_score(model, tok, pcfg, it, max_new_tokens=256)
    print("="*80)
    print("Q:", it["question"][:200])
    print("gold:", it["gold"], "| predicted:", rec["predicted"], "| P_clean:", rec["P_clean"])
    print("trace tail:", rec["trace"][-300:])
